In [0]:
/*
Project: Adventure Works Analytics
Purpose: Identify columns and data types from supporting tables required for dimensional modeling
Platform: Databricks SQL
Catalog: fea_academy
Schema: adventure_works
*/

-- =====================================================
-- Supporting Tables - Column Inventory
-- =====================================================

SELECT
    table_name,
    column_name,
    data_type
FROM `fea_academy`.information_schema.columns
WHERE table_schema = 'adventure_works'
  AND table_name IN (
      'sales_store',
      'production_productsubcategory',
      'production_productcategory',
      'sales_salesterritory',
      'sales_salesperson',
      'humanresources_employee'
  )
ORDER BY table_name, ordinal_position;

/*
Notes:
DIMENSIONAL ARCHITECTURE:
    - fact_sales -> One row per item sold
    - dim_date -> Calendar
    - dim_product -> Product + subcategory + category -> It will have the category hierarchy incorporated, avoiding unnecessary separate dimensions
    - dim_customer -> Customer + person and/or store -> It will combine the attributes of individuals and stores, while preserving the three types of customers identified
    - dim_geography -> Address + city + state + country -> It will be used twice: for the delivery address and the billing address
    - dim_credit_card -> Card type -> it will use the card type, without the need to include the card number in the analytical model
    - dim_sales_reason -> Reasons for sale
    - dim_salesperson -> Salesperson + employee
    - dim_sales_territory -> Sales territory, if applicable (optional)
    - Auxiliary -> BRIDGE_ORDER_SALES_REASON (for sales_salesorderheadersalesreason -> N:N conections)
*/

-- =====================================================
-- SUPPORTING TABLES REFERENTIAL INTEGRITY
-- =====================================================

SELECT
    'Product -> Subcategory' AS relationship,
    COUNT(*) AS unmatched_records
FROM `fea_academy`.`adventure_works`.`production_product` p
LEFT JOIN `fea_academy`.`adventure_works`.`production_productsubcategory` ps
    ON p.productsubcategoryid = ps.productsubcategoryid
WHERE p.productsubcategoryid IS NOT NULL
  AND ps.productsubcategoryid IS NULL

UNION ALL

SELECT
    'Subcategory -> Category',
    COUNT(*)
FROM `fea_academy`.`adventure_works`.`production_productsubcategory` ps
LEFT JOIN `fea_academy`.`adventure_works`.`production_productcategory` pc
    ON ps.productcategoryid = pc.productcategoryid
WHERE pc.productcategoryid IS NULL

UNION ALL

SELECT
    'Salesperson -> Employee',
    COUNT(*)
FROM `fea_academy`.`adventure_works`.`sales_salesperson` sp
LEFT JOIN `fea_academy`.`adventure_works`.`humanresources_employee` e
    ON sp.businessentityid = e.businessentityid
WHERE e.businessentityid IS NULL

UNION ALL

SELECT
    'Store -> Salesperson',
    COUNT(*)
FROM `fea_academy`.`adventure_works`.`sales_store` s
LEFT JOIN `fea_academy`.`adventure_works`.`sales_salesperson` sp
    ON s.salespersonid = sp.businessentityid
WHERE s.salespersonid IS NOT NULL
  AND sp.businessentityid IS NULL

UNION ALL

SELECT
    'Salesperson -> Territory',
    COUNT(*)
FROM `fea_academy`.`adventure_works`.`sales_salesperson` sp
LEFT JOIN `fea_academy`.`adventure_works`.`sales_salesterritory` st
    ON sp.territoryid = st.territoryid
WHERE sp.territoryid IS NOT NULL
  AND st.territoryid IS NULL

UNION ALL

SELECT
    'Customer -> Store',
    COUNT(*)
FROM `fea_academy`.`adventure_works`.`sales_customer` c
LEFT JOIN `fea_academy`.`adventure_works`.`sales_store` s
    ON c.storeid = s.businessentityid
WHERE c.storeid IS NOT NULL
  AND s.businessentityid IS NULL;

/*
Notes:
    - Product -> Subcategory: 0 unmatched records
    - Subcategory -> Category: 0 unmatched records
    - Salesperson -> Employee: 0 unmatched records
    - Store -> Salesperson: 0 unmatched records
    - Salesperson -> Territory: 0 unmatched records
    - Customer -> Store: 0 unmatched records
RESULT:
    - All six supporting relationships passed referential integrity checks
    - No unmatched records were identified
    - The validated relationships can be used in dimensional modeling
*/